# Cleaning Data Programmatically

In [1]:
import pandas as pd
import numpy as np

# Datasets context

# OEWS data (uncleaned )

The OEWS dataset was gathered manually as a CSV from the U.S. Bureau of Labor Statistics' website. The data was narrowed down to specifically focus on the managerial domain.

The dataset has a number of variables - there are four variables of significance to us:

- AREA_TITLE: Area/location name, e.g. Alabama
- OCC_CODE: The Standard Occupational Classification (SOC) code, e.g. 11-0000
- OCC_TITLE: The Standard Occupational Classification (SOC) title, e.g. Management Occupations
- H_MEAN: The mean hourly wage of the worker, e.g. 61.13

**Legend**:
- `*` indicates that a wage estimate is not available
- `**` indicates an employement estimate is not available
- `#` indicates that a wage is equal to or greater than 100 dollars per hour or greater than 280,000 dollars per year
- `~` indicates a percent total less than 0.05%

### PUMS data (cleaned)

The PUMS dataset was downloaded via the Census Data API from the United Statest Census Bureau, and narrowed down for the Kern County - Bakersfield MSA, California area.

Dataset variables:

- WRK: Whether the individual worked last week.
    - 0: N/A (not reported)
    - 1: Worked
    - 2: Did not work
- SEX: Sex (Male / Female) of the individual
    - 1: Male
    - 2: Female 
- SCOP: Standard Occupational Classification (SOC) codes for 2018 and later, based on the 2018 SOC codes

In [2]:
oews_data = pd.read_excel("Downloads/oes_research_2021_sec_55-56.xlsx")
oews_data.head()

,AREA,AREA_TITLE,NAICS,NAICS_TITLE,I_GROUP,OCC_CODE,OCC_TITLE,O_GROUP,TOT_EMP,EMP_PRSE,...,H_MEDIAN,H_PCT75,H_PCT90,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,ANNUAL,HOURLY
0,1,Alabama,55,Management of Companies and Enterprises,sector,00-0000,All Occupations,total,21920,0,...,35.6,56.94,79.49,35470,47040,74050,118440,165330,NaN,NaN
1,1,Alabama,55,Management of Companies and Enterprises,sector,11-0000,Management Occupations,major,4820,4.1,...,61.13,92.03,#,61600,94020,127140,191420,#,NaN,NaN
2,1,Alabama,55,Management of Companies and Enterprises,sector,11-1021,General and Operations Managers,detailed,1600,7,...,60.5,#,#,60010,78520,125850,#,#,NaN,NaN
3,1,Alabama,55,Management of Companies and Enterprises,sector,11-2021,Marketing Managers,detailed,140,13.6,...,61.13,99.23,#,65240,98680,127140,206410,#,NaN,NaN
4,1,Alabama,55,Management of Companies and Enterprises,sector,11-2022,Sales Managers,detailed,140,14.7,...,49.56,77.94,#,59390,79010,103080,162110,#,NaN,NaN


In [3]:
cleaned_pums = pd.read_csv("Downloads/cleaned_pums_2021.csv")
cleaned_pums.head()

,WRK,SEX,SOCP
0,1,2,119151
1,2,1,119111
2,1,2,113121
3,1,1,1110XX
4,1,1,113051


# Cleaning Data Tidiness Issue

In [4]:
cleaned_wage = oews_data.copy()

In [5]:
#Filter the dataframe for specific data elements
cleaned_wage = cleaned_wage[["AREA_TITLE", "OCC_CODE", "OCC_TITLE", "H_MEAN"]]

#Filter the dataframe for the AREA_TITLE to only apply to california 
cleaned_wage = cleaned_wage[cleaned_wage["AREA_TITLE"] == "California"]

cleaned_wage.describe()

,AREA_TITLE,OCC_CODE,OCC_TITLE,H_MEAN
count,3223,3223,3223,3223.00
unique,1,436,436,1715.00
top,California,00-0000,All Occupations,18.89
freq,3223,22,22,13.00


In [6]:
cleaned_wage.head()

,AREA_TITLE,OCC_CODE,OCC_TITLE,H_MEAN
377,California,00-0000,All Occupations,50.16
378,California,11-0000,Management Occupations,82.61
379,California,11-1011,Chief Executives,129.7
380,California,11-1021,General and Operations Managers,87.11
381,California,11-2011,Advertising and Promotions Managers,74.67


In [7]:
#Resetting the index
cleaned_wage = cleaned_wage.reset_index(drop=True)

cleaned_wage.head()

,AREA_TITLE,OCC_CODE,OCC_TITLE,H_MEAN
0,California,00-0000,All Occupations,50.16
1,California,11-0000,Management Occupations,82.61
2,California,11-1011,Chief Executives,129.7
3,California,11-1021,General and Operations Managers,87.11
4,California,11-2011,Advertising and Promotions Managers,74.67


# Cleaning the Data quality issue 

In [8]:
cleaned_wage.dtypes

AREA_TITLE    object
OCC_CODE      object
OCC_TITLE     object
H_MEAN        object
dtype: object

In [9]:
#1. Replace the * sign with np.nan
cleaned_wage['H_MEAN'] = cleaned_wage["H_MEAN"].replace({"*":np.nan})

#2. Deal with the outliers
cleaned_wage["H_MEAN"] = cleaned_wage["H_MEAN"].replace({"#":np.nan})

#3. Drop the NA values
cleaned_wage.dropna(inplace=True)

#4. Assert the number of NA values is 0
assert cleaned_wage.isna().sum().sum() == 0

/var/folders/hp/7w8yzdg13ng6crzbbv2st73w0000gn/T/ipykernel_73280/1762496174.py:5: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  cleaned_wage["H_MEAN"] = cleaned_wage["H_MEAN"].replace({"#":np.nan})


In [10]:
assert cleaned_wage.H_MEAN.dtypes == "float64"

# Consistency 

In [11]:
#Creating a new column
cleaned_pums["OCC_CODE"] = cleaned_pums["SOCP"]

In [12]:
#Replacing hyphen in occ_code variable in OECS
cleaned_wage["OCC_CODE"] = cleaned_wage["OCC_CODE"].str.replace("-", "", regex=False)

# Combine Datasets 

In [13]:
merged_df = pd.merge(cleaned_pums, cleaned_wage, on=["OCC_CODE"], how ="right")
merged_df.head()

,WRK,SEX,SOCP,OCC_CODE,AREA_TITLE,OCC_TITLE,H_MEAN
0,NaN,NaN,NaN,000000,California,All Occupations,50.16
1,NaN,NaN,NaN,110000,California,Management Occupations,82.61
2,NaN,NaN,NaN,111011,California,Chief Executives,129.70
3,1.0,1.0,111021,111021,California,General and Operations Managers,87.11
4,1.0,1.0,111021,111021,California,General and Operations Managers,87.11


In [14]:
#Drop NA 
merged_df = merged_df.dropna()

#Drop unnecssary columns SOCP and AREA TITLE
merged_df = merged_df.drop(["SOCP", "AREA_TITLE"], axis=1)

#Reset the index
merged_df = merged_df.reset_index(drop=True)

merged_df.head()

,WRK,SEX,OCC_CODE,OCC_TITLE,H_MEAN
0,1.0,1.0,111021,General and Operations Managers,87.11
1,1.0,1.0,111021,General and Operations Managers,87.11
2,1.0,2.0,111021,General and Operations Managers,87.11
3,1.0,2.0,111021,General and Operations Managers,87.11
4,1.0,1.0,111021,General and Operations Managers,87.11


# Store Data

In [15]:
merged_df.to_csv("cleaned_merged_df.csv", index=False)